# The Binomial Coefficient, Derived

The previous notebook used $\binom{n}{k}$; this one **derives** it from first principles — ordered selections, the factorial, and the division that removes duplicate orderings — then rebuilds the binomial PMF for a biased coin from scratch. It is based on the fourth video of Lesson 2 (Probability Distributions).


## 1. Introduction

How many ways are there to pick $k$ things out of $n$ when the order doesn't matter? This number — the binomial coefficient — is the counting engine inside the binomial distribution, and deriving it carefully pays off across all of probability and combinatorics.

After completing this notebook you will be able to:

- Count **ordered** selections with the falling product $n(n-1)\cdots(n-k+1)$.
- Explain why $k!$ is the number of orderings of $k$ objects.
- Derive $\binom{n}{k} = \frac{n!}{k!\,(n-k)!}$ and special cases like $\binom{n}{0} = 1$.
- Assemble the binomial PMF for a biased coin, where outcomes are no longer equally likely.


## 2. Theory

### Step 1: ordered selections

Pick $k$ numbers out of $n$, one at a time:

- 1st pick: $n$ options;
- 2nd pick: $n - 1$ options (one is taken);
- 3rd pick: $n - 2$ options;
- ... down to the $k$-th pick: $n - (k-1)$ options.

The picks multiply (each choice is free given the previous ones), so the number of **ordered** sets of length $k$ is

$$
n \cdot (n-1) \cdot (n-2) \cdots (n-k+1)
$$

### Step 2: how many orderings does each set have?

Each *unordered* set was counted many times — once per ordering. Ordering $k$ objects is the same kind of problem: $k$ options for the first position, $k-1$ for the second, ... , 1 for the last:

$$
k! = k \cdot (k-1) \cdots 2 \cdot 1
$$

$k!$ ("k factorial") is simply **the number of ways to order $k$ different objects** — e.g. $4! = 24$ orderings of a 4-element set.

### Step 3: divide

Since we want *unordered* sets, divide the ordered count by the $k!$ repeats:

$$
\binom{n}{k} = \frac{n(n-1)\cdots(n-k+1)}{k!} = \frac{n!}{k!\,(n-k)!}
$$

The second form follows because $n! = \underbrace{n \cdots (n-k+1)}_{\text{numerator}} \cdot \underbrace{(n-k) \cdots 1}_{(n-k)!}$ — dividing $n!$ by $(n-k)!$ leaves exactly the falling product.

### Special case: choosing nothing

$$
\binom{5}{0} = \frac{5!}{5!\,0!} = 1 \qquad (\text{with } 0! = 1)
$$

There is exactly **one** way to pick zero things: don't pick any. In general $\binom{n}{0} = 1$ for every $n$.


## 3. Worked Example: the biased coin

For a fair coin, all $2^5 = 32$ outcomes of five flips are equally likely, so probabilities are just counts over 32. **A biased coin breaks this**: outcomes now have different probabilities.

Say the coin lands heads with probability $0.3$ and tails with $0.7$:

- $HHHHH$: $0.3^5 = 0.00243$;
- $HHTTT$: $0.3^2 \cdot 0.7^3 = 0.03087$ — *higher*, because tails are more likely;
- in general, any specific sequence with $k$ heads: $0.3^k \cdot 0.7^{5-k}$.

Counts alone no longer give probabilities — but all sequences with the same number of heads still share one probability. So multiply by the number of orderings:

$$
P(X = k) = \binom{5}{k} \, (0.3)^k \, (0.7)^{5-k}
$$

That is the binomial distribution, rebuilt from its two ingredients: **per-sequence probability** (product rule) times **number of sequences** (binomial coefficient).


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb, factorial, perm

rng = np.random.default_rng(seed=42)


### The three counting steps, in code


In [ ]:
n, k = 5, 2

ordered = perm(n, k)                  # n * (n-1) * ... * (n-k+1)
orderings_per_set = factorial(k)      # k!
unordered = ordered // orderings_per_set

print(f"Ordered selections of {k} out of {n}:  {ordered}")
print(f"Orderings of each set (k!):         {orderings_per_set}")
print(f"Unordered sets = {ordered}/{orderings_per_set} = {unordered}  "
      f"(comb({n},{k}) = {comb(n, k)})")
print(f"Factorial identity: 5!/(2!·3!) = "
      f"{factorial(5) // (factorial(2) * factorial(3))}")
print(f"Choosing nothing: comb(5, 0) = {comb(5, 0)}   (0! = {factorial(0)})")


### Verifying k! by brute force

`itertools.permutations` lists every ordering of a 4-element set — there should be $4! = 24$.


In [ ]:
from itertools import permutations

orderings = list(permutations([1, 2, 3, 4]))
print(f"Orderings of {{1,2,3,4}}: {len(orderings)} = 4! = {factorial(4)}")
print("First five:", orderings[:5])


### The biased-coin PMF, assembled from its ingredients


In [ ]:
p = 0.3
print("k   per-sequence prob   x  #orderings  =  P(X=k)")
total = 0.0
for k in range(6):
    seq_prob = p ** k * (1 - p) ** (5 - k)
    pk = comb(5, k) * seq_prob
    total += pk
    print(f"{k}      {seq_prob:.5f}          x  {comb(5, k):>2}          {pk:.5f}")
print(f"Sum: {total:.4f}")


### Simulation check


In [ ]:
n_runs = 200_000
heads = (rng.random((n_runs, 5)) < 0.3).sum(axis=1)

print("k   simulated   formula")
for k in range(6):
    formula = comb(5, k) * 0.3 ** k * 0.7 ** (5 - k)
    print(f"{k}     {(heads == k).mean():.4f}     {formula:.4f}")


## 5. Visualizations

### Ordered vs. unordered: the k! collapse

All 20 ordered picks of 2 numbers out of {1..5}, grouped into the 10 unordered pairs — each pair appears $2! = 2$ times.


In [ ]:
from itertools import permutations as perms

fig, ax = plt.subplots(figsize=(9, 4))

pairs = {}
for a, b in perms(range(1, 6), 2):
    key = tuple(sorted((a, b)))
    pairs.setdefault(key, []).append((a, b))

for i, (key, ordered_versions) in enumerate(sorted(pairs.items())):
    for j, (a, b) in enumerate(ordered_versions):
        ax.add_patch(plt.Rectangle((i - 0.42, j), 0.84, 0.9,
                                   facecolor="#4e79a7" if j == 0 else "#9ecae1",
                                   edgecolor="white"))
        ax.text(i, j + 0.45, f"({a},{b})", ha="center", va="center",
                color="white", fontsize=10)

ax.set_xlim(-0.8, 9.8)
ax.set_ylim(0, 2.6)
ax.set_xticks(range(10))
ax.set_xticklabels(["{%d,%d}" % k for k in sorted(pairs)], fontsize=9)
ax.set_yticks([])
ax.set_title("20 ordered picks ÷ 2! orderings = 10 unordered sets = C(5,2)")
plt.show()


### Fair vs. biased, per-sequence and grouped

Left: for a fair coin every sequence has probability 1/32, so the PMF is counts/32. Right: for the biased coin the per-sequence probability itself depends on k — the PMF is the product of the two curves.


In [ ]:
ks = np.arange(6)
counts = np.array([comb(5, k) for k in ks])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(ks, counts / 32, color="#4e79a7", edgecolor="white")
axes[0].set_title("Fair coin: PMF = C(5,k) / 32")
axes[0].set_xlabel("k (heads)")
axes[0].set_ylabel("P(X = k)")

seq_prob = 0.3 ** ks * 0.7 ** (5 - ks)
axes[1].bar(ks, counts * seq_prob, color="#f28e2b", edgecolor="white",
            label="PMF = C(5,k) · seq. prob")
axes[1].plot(ks, seq_prob, "o--", color="gray", label="per-sequence prob")
axes[1].set_title("Biased coin (p = 0.3): counts × unequal sequence probs")
axes[1].set_xlabel("k (heads)")
axes[1].legend()

for ax in axes:
    ax.grid(True, alpha=0.3, axis="y")
    ax.set_xticks(ks)
plt.tight_layout()
plt.show()


## 6. Exercises

### Basic

1. How many ways can 3 students be chosen from a class of 10 for a committee (order doesn't matter)? And for president/secretary/treasurer roles (order matters)?
2. Why is $\binom{n}{n} = 1$? Give both the formula argument and the counting argument.

### Intermediate

3. A lottery draws 6 numbers out of 49, unordered. How many possible tickets are there, and what is the probability of winning with one ticket?
4. For the biased coin ($p = 0.3$, 5 flips): which value of $k$ has the highest probability? Would that change with $p = 0.7$?

### Advanced

5. Show numerically that $\sum_{k=0}^{n} \binom{n}{k} = 2^n$ for several $n$, and explain why: what do the left and right sides each count?


## 7. Solutions


In [ ]:
# Exercise 1
print(f"1.  Committee: comb(10,3) = {comb(10, 3)}; "
      f"roles: perm(10,3) = {perm(10, 3)}")

# Exercise 2
print("2.  Formula: n!/(n! 0!) = 1. Counting: taking everything is one single way.")

# Exercise 3
tickets = comb(49, 6)
print(f"3.  comb(49,6) = {tickets:,} tickets -> P(win) = {1/tickets:.2e}")

# Exercise 4
pmf_03 = [comb(5, k) * 0.3**k * 0.7**(5-k) for k in range(6)]
pmf_07 = [comb(5, k) * 0.7**k * 0.3**(5-k) for k in range(6)]
print(f"4.  p=0.3: argmax k = {int(np.argmax(pmf_03))}; "
      f"p=0.7: argmax k = {int(np.argmax(pmf_07))} — the peak mirrors.")

# Exercise 5
for n in [3, 5, 8, 12]:
    total = sum(comb(n, k) for k in range(n + 1))
    print(f"5.  n={n}: sum of C(n,k) = {total} = 2^{n} = {2**n}")
print("    Left: sequences grouped by number of heads. "
      "Right: all length-n H/T sequences. Same set, counted two ways.")


## 8. Conclusion

- Ordered selections: $n(n-1)\cdots(n-k+1)$. Orderings of a $k$-set: $k!$. Unordered selections: their quotient, $\binom{n}{k} = \frac{n!}{k!(n-k)!}$.
- $0! = 1$ and $\binom{n}{0} = \binom{n}{n} = 1$: there is exactly one way to take nothing (or everything).
- For a **biased** coin, outcomes are not equally likely, so probabilities are no longer counts over a total — but sequences with the same $k$ still share the probability $p^k(1-p)^{n-k}$, and multiplying by $\binom{n}{k}$ rebuilds the binomial PMF.
- Counting arguments (like $\sum_k \binom{n}{k} = 2^n$) are proofs by "same set, counted two ways" — a recurring trick in probability.
